In [2]:
"""
Retrieval Comparison: BM25 vs TF-IDF vs Semantic Similarity
============================================================
Jalankan script ini untuk melihat perbedaan konteks yang di-retrieve
oleh masing-masing metode.
 
Output: retrieval_results.json  →  buka dengan viewer HTML
"""

import json
import logging
from pathlib import Path
 
from langchain_community.document_loaders import JSONLoader
from langchain_huggingface.embeddings import HuggingFaceEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain.retrievers import BM25Retriever
 
# Sklearn untuk TF-IDF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
 
logging.basicConfig(level=logging.INFO, format="%(asctime)s [%(levelname)s] %(message)s")
logger = logging.getLogger(__name__)

In [3]:
CONFIG = {
    "json_path": "/Users/muhammadzuamaalamin/Documents/RisetTextMining/Qwen_json_20260313_krtrbraab.json",
    "embedding_model": "/Users/muhammadzuamaalamin/Documents/fintunellm/model/bge-m3",
    "faiss_index_path": "faiss_index",
    "chunk_size": 500,
    "chunk_overlap": 50,
    "top_k": 5,
    "output_json": "retrieval_results.json",
}

In [4]:
 
# Query yang ingin dibandingkan
QUERIES = [
    "Apa hak-hak warga negara yang dijamin dalam UUD 1945?",
    "Bunyi pasal 1 UUD 1945 tentang bentuk negara",
    "Bagaimana sistem pemerintahan presiden menurut konstitusi?",
    "Hak atas pendidikan bagi warga negara Indonesia",
]
 

In [5]:
# ---------------------------------------------------------------------------
# Load & split
# ---------------------------------------------------------------------------
 
def load_and_split(cfg: dict) -> tuple[list, list]:
    loader = JSONLoader(file_path=cfg["json_path"], jq_schema=".[]", text_content=False)
    docs = loader.load()
    logger.info("Memuat %d dokumen", len(docs))
 
    splitter = RecursiveCharacterTextSplitter(
        chunk_size=cfg["chunk_size"],
        chunk_overlap=cfg["chunk_overlap"],
        separators=["\n\n", "\n", ". ", " ", ""],
    )
    chunks = splitter.split_documents(docs)
    logger.info("Menghasilkan %d chunks", len(chunks))
    return docs, chunks
 

In [6]:

# ---------------------------------------------------------------------------
# TF-IDF Retriever (custom, tidak ada di LangChain default)
# ---------------------------------------------------------------------------
 
class TFIDFRetriever:
    """TF-IDF retriever sederhana menggunakan sklearn."""
 
    def __init__(self, chunks: list, k: int = 5):
        self.chunks = chunks
        self.k = k
        self.texts = [c.page_content for c in chunks]
 
        logger.info("Membangun TF-IDF matrix...")
        self.vectorizer = TfidfVectorizer(
            analyzer="word",
            ngram_range=(1, 2),   # unigram + bigram
            min_df=1,
            sublinear_tf=True,    # log normalization
        )
        self.tfidf_matrix = self.vectorizer.fit_transform(self.texts)
        logger.info("TF-IDF matrix shape: %s", self.tfidf_matrix.shape)
 
    def retrieve(self, query: str) -> list[dict]:
        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.tfidf_matrix).flatten()
        top_indices = np.argsort(scores)[::-1][: self.k]
 
        results = []
        for idx in top_indices:
            results.append({
                "content": self.texts[idx],
                "score": float(scores[idx]),
                "metadata": self.chunks[idx].metadata,
            })
        return results
 
# ---------------------------------------------------------------------------
# BM25 Retriever wrapper
# ---------------------------------------------------------------------------
 
class BM25RetrieverWrapper:
    def __init__(self, chunks: list, k: int = 5):
        self.k = k
        self.retriever = BM25Retriever.from_documents(chunks)
        self.retriever.k = k
        logger.info("BM25 retriever siap")
 
    def retrieve(self, query: str) -> list[dict]:
        # invoke() menggantikan get_relevant_documents() yang deprecated
        docs = self.retriever.invoke(query)
        return [
            {
                "content": d.page_content,
                "score": None,   # BM25 LangChain tidak expose score langsung
                "metadata": d.metadata,
            }
            for d in docs
        ]
 

In [7]:

class SemanticRetriever:
    def __init__(self, chunks: list, cfg: dict):
        self.k = cfg["top_k"]
        embeddings = HuggingFaceEmbeddings(
            model_name=cfg["embedding_model"],
            model_kwargs={"device": "cpu"},
        )
        faiss_path = cfg["faiss_index_path"]
        if Path(faiss_path).exists():
            logger.info("Memuat FAISS dari: %s", faiss_path)
            self.vectorstore = FAISS.load_local(
                faiss_path, embeddings, allow_dangerous_deserialization=True
            )
        else:
            logger.info("Membuat FAISS index baru...")
            self.vectorstore = FAISS.from_documents(chunks, embeddings)
            self.vectorstore.save_local(faiss_path)
        logger.info("Semantic retriever siap")
 
    def retrieve(self, query: str) -> list[dict]:
        results = self.vectorstore.similarity_search_with_score(query, k=self.k)
        return [
            {
                "content": doc.page_content,
                "score": float(score),
                "metadata": doc.metadata,
            }
            for doc, score in results
        ]
 

In [8]:

def highlight_keywords(text: str, query: str) -> list[str]:
    """Kembalikan list kata dari query yang muncul di text (lowercase)."""
    query_words = set(query.lower().split())
    found = [w for w in query_words if w in text.lower() and len(w) > 3]
    return found
 
# ---------------------------------------------------------------------------
# Jalankan perbandingan & simpan JSON
# ---------------------------------------------------------------------------
 
def run_comparison(cfg: dict, queries: list[str]) -> dict:
    _, chunks = load_and_split(cfg)
 
    logger.info("Membangun semua retriever...")
    bm25_ret = BM25RetrieverWrapper(chunks, k=cfg["top_k"])
    tfidf_ret = TFIDFRetriever(chunks, k=cfg["top_k"])
    semantic_ret = SemanticRetriever(chunks, cfg)
 
    all_results = {}
 
    for query in queries:
        logger.info("Query: %s", query)
        query_results = {}
 
        for method_name, retriever in [
            ("bm25", bm25_ret),
            ("tfidf", tfidf_ret),
            ("semantic", semantic_ret),
        ]:
            raw = retriever.retrieve(query)
            processed = []
            for rank, item in enumerate(raw, 1):
                processed.append({
                    "rank": rank,
                    "content": item["content"],
                    "score": item["score"],
                    "score_label": f"{item['score']:.4f}" if item["score"] is not None else "N/A",
                    "keywords_found": highlight_keywords(item["content"], query),
                    "char_count": len(item["content"]),
                })
            query_results[method_name] = processed
 
        all_results[query] = query_results
 
    output = {
        "queries": queries,
        "methods": ["bm25", "tfidf", "semantic"],
        "method_labels": {
            "bm25": "BM25",
            "tfidf": "TF-IDF",
            "semantic": "Semantic (FAISS + BGE-M3)",
        },
        "method_descriptions": {
            "bm25": "Keyword matching berbasis frekuensi kata, dengan normalisasi panjang dokumen. Unggul untuk query eksak / istilah spesifik.",
            "tfidf": "Term Frequency–Inverse Document Frequency dengan bigram. Memberi bobot lebih pada kata langka di corpus. Mirip BM25 tapi tanpa normalisasi panjang.",
            "semantic": "Dense vector embedding (BGE-M3) + FAISS cosine similarity. Menangkap makna semantik walau kata berbeda.",
        },
        "results": all_results,
    }
 
    output_path = cfg["output_json"]
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(output, f, ensure_ascii=False, indent=2)
 
    logger.info("Hasil disimpan ke: %s", output_path)
    return output
 

In [9]:
def print_comparison(data: dict, query_index: int = 0):
    query = data["queries"][query_index]
    print(f"\n{'='*70}")
    print(f"QUERY: {query}")
    print(f"{'='*70}")
 
    for method in data["methods"]:
        label = data["method_labels"][method]
        desc  = data["method_descriptions"][method]
        results = data["results"][query][method]
 
        print(f"\n{'─'*70}")
        print(f"[{label}]  {desc}")
        print(f"{'─'*70}")
 
        for item in results:
            score_str = f"score={item['score_label']}" if item["score_label"] != "N/A" else "score=N/A"
            kw = ", ".join(item["keywords_found"]) if item["keywords_found"] else "-"
            print(f"\n  Rank {item['rank']} | {score_str} | kata query: [{kw}]")
            print(f"  {item['content'][:350].strip()}")
            if len(item["content"]) > 350:
                print("  ...")
 
 
if __name__ == "__main__":
    data = run_comparison(CONFIG, QUERIES)
 
    # Tampilkan preview untuk query pertama
    print_comparison(data, query_index=0)
 
    print(f"\n\nSemua hasil tersimpan di: {CONFIG['output_json']}")
    print("Buka retrieval_viewer.html di browser untuk tampilan visual lengkap.")


2026-03-16 10:48:47,996 [INFO] Memuat 71 dokumen
2026-03-16 10:48:47,999 [INFO] Menghasilkan 73 chunks
2026-03-16 10:48:47,999 [INFO] Membangun semua retriever...
2026-03-16 10:48:48,002 [INFO] BM25 retriever siap
2026-03-16 10:48:48,003 [INFO] Membangun TF-IDF matrix...
2026-03-16 10:48:48,009 [INFO] TF-IDF matrix shape: (73, 1417)


AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

AttributeError: 'MessageFactory' object has no attribute 'GetPrototype'

2026-03-16 10:48:51,881 [INFO] TensorFlow version 2.14.0 available.
2026-03-16 10:48:52,220 [INFO] Load pretrained SentenceTransformer: /Users/muhammadzuamaalamin/Documents/fintunellm/model/bge-m3
2026-03-16 10:48:53,390 [INFO] Membuat FAISS index baru...
2026-03-16 10:48:57,694 [INFO] Loading faiss.
2026-03-16 10:48:57,724 [INFO] Successfully loaded faiss.
2026-03-16 10:48:57,733 [INFO] Semantic retriever siap
2026-03-16 10:48:57,733 [INFO] Query: Apa hak-hak warga negara yang dijamin dalam UUD 1945?
2026-03-16 10:48:57,853 [INFO] Query: Bunyi pasal 1 UUD 1945 tentang bentuk negara
2026-03-16 10:48:57,981 [INFO] Query: Bagaimana sistem pemerintahan presiden menurut konstitusi?
2026-03-16 10:48:58,109 [INFO] Query: Hak atas pendidikan bagi warga negara Indonesia
2026-03-16 10:48:58,238 [INFO] Hasil disimpan ke: retrieval_results.json



QUERY: Apa hak-hak warga negara yang dijamin dalam UUD 1945?

──────────────────────────────────────────────────────────────────────
[BM25]  Keyword matching berbasis frekuensi kata, dengan normalisasi panjang dokumen. Unggul untuk query eksak / istilah spesifik.
──────────────────────────────────────────────────────────────────────

  Rank 1 | score=N/A | kata query: [warga, negara, dalam]
  {"id": "psl_30_ay_1", "pasal": 30, "ayat": 1, "bab": "BAB XII - PERTAHANAN NEGARA", "content": "Tiap-tiap warga negara berhak dan wajib ikut serta dalam usaha pembelaan negara."}

  Rank 2 | score=N/A | kata query: [hak-hak, negara, dalam, yang]
  {"id": "psl_18_ay_1", "pasal": 18, "ayat": 1, "bab": "BAB VI - PEMERINTAHAN DAERAH", "content": "Pembagian daerah Indonesia atas daerah besar dan kecil, dengan bentuk susunan pemerintahannya ditetapkan dengan undang-undang, dengan memandang dan mengingati dasar permusyawaratan dalam sistem pemerintahan negara, dan hak-hak asal-usul dalam daerah-dae
  ..